# Qwen-VL All-Val Evaluation on Kaggle (via Vertex AI)

This notebook is a **thin wrapper** around the `aiforensics` CLI to evaluate the
`qwen_vl` baseline via a Google Cloud Vertex AI Dedicated Endpoint across **all**
generators found under `DATA_ROOT` using **only** the dataset `val` split.

It does **not** train any model, and does **not** run CLIP, NPR, or Assisted Qwen.

Key features:
- Reads credentials securely from the Kaggle Secret `GOOGLE_APPLICATION_CREDENTIALS`.
- Uses Dedicated Endpoint domain `*.prediction.vertexai.goog`.
- Discovers all generator directories under `<DATA_ROOT>/<generator>/val/{ai,nature}/*`.
- Configurable `MAX_IMAGES_PER_GENERATOR = 0` (0 evaluates all available val images).
- Writes inspectable artifacts, metrics, and report under `/kaggle/working/outputs-qwen-all-val/`.

## 1. Runtime preflight

This cell only reports what the environment looks like. It deliberately does
**not** fail when the notebook kernel is newer than Python 3.10: the kernel never
imports `aiforensics`, the CLI does. What matters is whether a Python 3.10
interpreter is available for the CLI, because `pyproject.toml` declares

```text
requires-python = ">=3.10,<3.11"
```

GPU output below is informational. Real device selection and deferral stay
inside the baseline adapters.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# User-editable: where the Python 3.10 environment for the CLI lives.
CLI_VENV_PATH = Path("/kaggle/working/aiforensics-venv310")

TARGET_PY = (3, 10)


def _venv_bin(venv_path: Path) -> Path:
    """Return the scripts directory of a virtual environment."""
    return venv_path / ("Scripts" if os.name == "nt" else "bin")


def _interpreter_version(executable: str) -> tuple[int, int] | None:
    """Return (major, minor) for an interpreter, or None when unusable."""
    try:
        result = subprocess.run(
            [executable, "-c", "import sys; print(sys.version_info[0], sys.version_info[1])"],
            capture_output=True,
            text=True,
            check=False,
        )
    except OSError:
        return None
    if result.returncode != 0:
        return None
    parts = result.stdout.split()
    if len(parts) != 2:
        return None
    return int(parts[0]), int(parts[1])


def find_cli_python() -> str | None:
    """Find an interpreter that satisfies the repository Python contract."""
    candidates = [
        str(_venv_bin(CLI_VENV_PATH) / "python"),
        shutil.which("python3.10"),
        sys.executable,
    ]
    for candidate in candidates:
        if not candidate or not Path(candidate).exists():
            continue
        if _interpreter_version(candidate) == TARGET_PY:
            return candidate
    return None


print("notebook kernel:", sys.version.split()[0], "(informational only)")
print("working directory:", Path.cwd())

CLI_PYTHON = find_cli_python()
if CLI_PYTHON:
    print("Python 3.10 for the CLI:", CLI_PYTHON)
else:
    print(
        "No Python 3.10 interpreter found yet.\n"
        "Run the optional provisioning cell in section 2 before installing the package."
    )

gpu = shutil.which("nvidia-smi")
if gpu:
    subprocess.run([gpu], check=False)
else:
    print("nvidia-smi not found: no GPU visible to this runtime (informational).")

## 2. Optional: provision Python 3.10 for the CLI

Run this section **only when section 1 reported no Python 3.10 interpreter**.

It creates a dedicated virtual environment on Python 3.10 and puts it first on
`PATH`, so later cells can call `aiforensics` unchanged. This satisfies the
repository's `requires-python` contract honestly: the package is installed under
a real 3.10 interpreter. Never edit `pyproject.toml` to make an install succeed.

This step needs **network access** (to fetch `uv`, the interpreter, and the
dependencies).

In [ ]:
def provision_cli_python(venv_path: Path) -> str:
    """Create a Python 3.10 virtual environment and prepend it to PATH."""
    if shutil.which("uv") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "uv"],
            check=True,
        )

    uv = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
    subprocess.run([uv, "python", "install", "3.10"], check=True)
    subprocess.run(
        [uv, "venv", "--seed", "--no-project", "--python", "3.10", str(venv_path)],
        check=True,
    )

    bin_dir = _venv_bin(venv_path)
    os.environ["PATH"] = f"{bin_dir}{os.pathsep}{os.environ.get('PATH', '')}"
    os.environ["VIRTUAL_ENV"] = str(venv_path)
    return str(bin_dir / "python")


CLI_PYTHON = provision_cli_python(CLI_VENV_PATH)
print("provisioned CLI interpreter:", CLI_PYTHON)

### 2b. Verify the CLI interpreter

This is the first hard gate. If no valid Python 3.10 environment exists after
provisioning, stop here and fix the environment instead of working around the
version contract.

In [ ]:
resolved = shutil.which("python") or ""
version = _interpreter_version(resolved) if resolved else None

print("python on PATH:", resolved or "<none>")
print("python version:", ".".join(str(p) for p in version) if version else "<unknown>")

if version != TARGET_PY:
    raise RuntimeError(
        "No usable Python 3.10 environment for the CLI. The repository requires "
        ">=3.10,<3.11. Run the provisioning cell above, or select a runtime that "
        "provides Python 3.10. Do not modify pyproject.toml to bypass this."
    )

print("OK: the CLI will run under Python 3.10.")

## 3. Repository location

Point `REPO_ROOT` at a checkout of this repository. Either attach it as a Kaggle
dataset/utility script and copy it into writable storage, or set `REPO_GIT_URL`
to a repository you control and let the cell clone it (needs Internet enabled).
Do not embed credentials in this notebook; use an environment variable or Kaggle
Secrets if a private clone needs authentication.

In [ ]:
# User-editable inputs. The defaults below are this project's own values so a
# fresh session needs no editing; override them for a fork or mirror.
REPO_ROOT = Path("/kaggle/working/ai-image-forensics")
REPO_GIT_URL = "https://github.com/Nnguyen-dev2805/ai-image-forensics.git"

if not REPO_ROOT.exists() and REPO_GIT_URL:
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_GIT_URL, str(REPO_ROOT)], check=True)

REQUIRED_REPO_FILES = [
    REPO_ROOT / "pyproject.toml",
    REPO_ROOT / "configs/phase_ab.yaml",
]
missing = [str(path) for path in REQUIRED_REPO_FILES if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "REPO_ROOT does not look like this repository. Missing: "
        + ", ".join(missing)
        + ". Set REPO_ROOT (or REPO_GIT_URL) to a valid checkout."
    )

os.chdir(REPO_ROOT)
print("repository root:", REPO_ROOT)

## 4. Install the package

Dependency names come from `pyproject.toml`; nothing is pinned again here. The
optional extras map to the baselines: `clip` for the CLIP probe, `qwen` for
Qwen-VL and Assisted Qwen, `npr` for the NPR runtime bridge.

Model weights and the NPR checkpoint are **not** packaged with the repository.

In [ ]:
os.environ["AIF_REPO_ROOT"] = str(REPO_ROOT)
print("AIF_REPO_ROOT =", os.environ["AIF_REPO_ROOT"])

In [ ]:
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

# 1. Locate the Python 3.10 CLI virtual environment provisioned in Section 2
cli_python = shutil.which("python")
cli_py_version = (
    subprocess.run([cli_python, "-V"], capture_output=True, text=True).stdout
    if cli_python
    else ""
)
if not cli_python or "3.10" not in cli_py_version:
    cli_python = str(Path("/kaggle/working/aiforensics-venv310/bin/python"))

print("CLI Python 3.10 target:", cli_python)

# 2. Install aiforensics with [vertex] extra into the Python 3.10 environment
subprocess.run(
    [cli_python, "-m", "pip", "install", "--quiet", "--upgrade", "pip"],
    check=True,
)
subprocess.run(
    [cli_python, "-m", "pip", "install", "-e", ".[vertex]"],
    cwd=str(REPO_ROOT),
    check=True,
)

# 3. Ensure the notebook kernel (Python 3.12) has openai and google dependencies for Section 8
try:
    import openai
    from google.auth.transport.requests import Request
    from google.oauth2 import service_account
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "openai", "google-auth", "requests"],
        check=True,
    )

print("aiforensics CLI and Vertex dependencies installed successfully!")

### 4b. Verify the CLI resolves from the Python 3.10 environment

Second hard gate: the `aiforensics` entry point must exist and run under the
interpreter verified in section 2b.

In [ ]:
cli_path = shutil.which("aiforensics")
print("aiforensics on PATH:", cli_path or "<none>")

if not cli_path:
    raise RuntimeError(
        "The aiforensics CLI is not on PATH. Re-run the install cell, and make "
        "sure the Python 3.10 environment from section 2 is still first on PATH."
    )

result = subprocess.run([cli_path, "--help"], capture_output=True, text=True, check=False)
if result.returncode != 0:
    raise RuntimeError(f"aiforensics --help failed with exit code {result.returncode}")

print("OK: CLI available at", cli_path)

## 5. Storage inputs

Kaggle separates **read-only** attached data from **writable** working storage:

- `/kaggle/input/<your-dataset>` is read-only. Research images live here.
- `/kaggle/working` is writable but ephemeral; save notebook output to keep artifacts.

This notebook evaluates Qwen-VL via Vertex AI across all generators in the dataset
`val` split only (`<DATA_ROOT>/<generator>/val/{ai,nature}/*`).

In [ ]:
# User-editable: Kaggle mount points.
KAGGLE_INPUT_ROOT = Path("/kaggle/input")  # read-only attached datasets
KAGGLE_WORKING_ROOT = Path("/kaggle/working")  # writable, ephemeral

# Full path to the attached dataset directory that holds the images.
# Expected GenImage layout: <DATA_ROOT>/<generator>/val/{ai,nature}/*
INPUT_DATA_DIR = KAGGLE_INPUT_ROOT / "datasets/yangsangtai/tiny-genimage"

# Maximum number of images per generator to evaluate.
# 0 means full val (all available val images for each generator).
# Set to a small integer (e.g. 10 or 50) for fast smoke/sanity checks.
MAX_IMAGES_PER_GENERATOR = 0

# Set True to build manifests from the val split of all generators in INPUT_DATA_DIR;
# set False when manifest CSV is already provisioned.
BUILD_MANIFESTS = True
INPUT_MANIFEST_DIR = KAGGLE_INPUT_ROOT / "<your-manifests-dataset>"

# Read-only inputs.
DATA_ROOT = INPUT_DATA_DIR

# Writable outputs: never place these under /kaggle/input.
CACHE_ROOT = KAGGLE_WORKING_ROOT / "cache"
OUTPUT_ROOT = Path("/kaggle/working/outputs-qwen-all-val")
EXTERNAL_ROOT = KAGGLE_WORKING_ROOT / "external"

# Built manifests must land in writable storage; provisioned ones stay read-only.
MANIFEST_ROOT = KAGGLE_WORKING_ROOT / "manifests" if BUILD_MANIFESTS else INPUT_MANIFEST_DIR

# Dedicated manifest for all-val evaluation
GENIMAGE_ALL_VAL_MANIFEST = MANIFEST_ROOT / "genimage_all_val.csv"

writable_roots = [CACHE_ROOT, OUTPUT_ROOT, EXTERNAL_ROOT]
if BUILD_MANIFESTS:
    writable_roots.append(MANIFEST_ROOT)
for writable in writable_roots:
    writable.mkdir(parents=True, exist_ok=True)

if not BUILD_MANIFESTS and MANIFEST_ROOT.is_relative_to(KAGGLE_INPUT_ROOT):
    print("manifest root is read-only; manifests must already exist there")

print("data root    : (read-only)", DATA_ROOT)
print("manifest root:", "(writable)" if BUILD_MANIFESTS else "(read-only)", MANIFEST_ROOT)
print("cache root   : (writable)", CACHE_ROOT)
print("output root  : (writable)", OUTPUT_ROOT)
print("external root: (writable)", EXTERNAL_ROOT)
print("max images/generator:", MAX_IMAGES_PER_GENERATOR, "(0 = full val)")
print("build manifests:", BUILD_MANIFESTS)

## 6. Generate the runtime config

`configs/qwen_vertex_all_val.yaml` is treated as a **read-only template**. This cell copies
it, discovers all generators with a `val` split under `DATA_ROOT`, and rewrites path and
generator values under `.cache/aiforensics-notebook/` inside the repository.

In [ ]:
import yaml

TEMPLATE_CONFIG = REPO_ROOT / "configs/qwen_vertex_all_val.yaml"
GENERATED_CONFIG = REPO_ROOT / ".cache/aiforensics-notebook/qwen_vertex_all_val_kaggle.yaml"

with open(TEMPLATE_CONFIG, encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

# Relocate paths to runtime storage
cfg["paths"]["data_root"] = str(DATA_ROOT)
cfg["paths"]["manifest_root"] = str(MANIFEST_ROOT)
cfg["paths"]["cache_root"] = str(CACHE_ROOT)
cfg["paths"]["output_root"] = str(OUTPUT_ROOT)
cfg["paths"]["external_root"] = str(EXTERNAL_ROOT)


def discover_generator_dirs(data_root: Path) -> list[str]:
    """Discover all generator directories that have a val split."""
    if not data_root.is_dir():
        return []
    found = []
    for entry in sorted(data_root.iterdir()):
        if entry.is_dir() and (entry / "val").is_dir():
            found.append(entry.name)
    return found


all_val_generators = discover_generator_dirs(DATA_ROOT)

cfg["datasets"]["genimage_unseen"]["generators"] = all_val_generators
cfg["datasets"]["genimage_unseen"]["manifest"] = str(GENIMAGE_ALL_VAL_MANIFEST)
cfg["datasets"]["genimage_unseen"]["source_split"] = "val"
cfg["datasets"]["genimage_unseen"]["max_images"] = int(MAX_IMAGES_PER_GENERATOR)

cfg["baselines"]["qwen_vl"]["provider"] = "vertex_openai"
cfg["report"]["filename"] = "qwen_vertex_all_val_report.md"

GENERATED_CONFIG.parent.mkdir(parents=True, exist_ok=True)
with open(GENERATED_CONFIG, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

os.environ["AIF_CONFIG"] = str(GENERATED_CONFIG)
print("runtime config:", GENERATED_CONFIG)
print("report filename:", cfg["report"]["filename"])
print("generators (val only):", all_val_generators)
print("max images per generator:", MAX_IMAGES_PER_GENERATOR)

## 7. Validate provisioned inputs

This cell checks the runtime paths and lists all discovered generator directories
with a `val` split.

In [ ]:
print("runtime config :", GENERATED_CONFIG, "exists:", GENERATED_CONFIG.is_file())
print("data root      :", DATA_ROOT, "exists:", DATA_ROOT.is_dir())
print("manifest root  :", MANIFEST_ROOT, "exists:", MANIFEST_ROOT.is_dir())
print("cache root     :", CACHE_ROOT, "exists:", CACHE_ROOT.is_dir())
print("output root    :", OUTPUT_ROOT, "exists:", OUTPUT_ROOT.is_dir())

found_val_generators = []
if DATA_ROOT.is_dir():
    for entry in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
        if (entry / "val").is_dir():
            found_val_generators.append(entry.name)

print("\ngenerator directories with val/ under data root:", len(found_val_generators))
for name in found_val_generators:
    print("  -", name)
if not found_val_generators and BUILD_MANIFESTS:
    print(
        "  none found: check DATA_ROOT. Expected "
        "<DATA_ROOT>/<generator>/val/<ai|nature>/"
    )

print(
    "\nall-val manifest target:",
    GENIMAGE_ALL_VAL_MANIFEST,
    "exists:",
    GENIMAGE_ALL_VAL_MANIFEST.is_file(),
)
if BUILD_MANIFESTS:
    print("  (BUILD_MANIFESTS is true: prepare --build-manifests will create/overwrite this)")

## 8. Vertex Qwen endpoint preflight

This validates the cloud Qwen path before the CLI runtime is changed.
Authentication comes from the Kaggle Secret named
`GOOGLE_APPLICATION_CREDENTIALS`. The secret value is the service-account JSON
payload. The notebook parses it in memory, refreshes Google credentials, and
passes the resulting bearer token to the OpenAI-compatible client.

Credential material is never printed.

In [ ]:
# AIF_SECTION: vertex_qwen
import json

from google.auth.transport.requests import Request
from google.oauth2 import service_account
from kaggle_secrets import UserSecretsClient
from openai import OpenAI

VERTEX_PROJECT_ID = "579187260419"
VERTEX_LOCATION = "asia-southeast1"
VERTEX_ENDPOINT_ID = "mg-endpoint-ccc259a2-c268-4ca6-8713-3bcdaaaf5909"
VERTEX_ENDPOINT_DOMAIN = (
    "mg-endpoint-ccc259a2-c268-4ca6-8713-3bcdaaaf5909."
    "asia-southeast1-635507464424.prediction.vertexai.goog"
)
VERTEX_MODEL_ID = "qwen2_5-vl-7b-instruct-1788570383931"
VERTEX_ENDPOINT_PATH = (
    "/v1/projects/579187260419/locations/asia-southeast1/endpoints/"
    "mg-endpoint-ccc259a2-c268-4ca6-8713-3bcdaaaf5909"
)
EXPECTED_BASE_URL = f"https://{VERTEX_ENDPOINT_DOMAIN}{VERTEX_ENDPOINT_PATH}"


def build_vertex_base_url(project_id: str, location: str, endpoint_id: str, domain: str) -> str:
    cleaned_domain = domain.removeprefix("https://").rstrip("/")
    if cleaned_domain == "aiplatform.googleapis.com":
        raise ValueError("Dedicated Endpoint runs must use the prediction.vertexai.goog domain")
    if not cleaned_domain.endswith(".prediction.vertexai.goog"):
        raise ValueError(f"Unexpected Vertex dedicated endpoint domain: {cleaned_domain}")
    return (
        f"https://{cleaned_domain}/v1/projects/{project_id}"
        f"/locations/{location}/endpoints/{endpoint_id}"
    )


BASE_URL = build_vertex_base_url(
    VERTEX_PROJECT_ID,
    VERTEX_LOCATION,
    VERTEX_ENDPOINT_ID,
    VERTEX_ENDPOINT_DOMAIN,
)
if BASE_URL != EXPECTED_BASE_URL:
    raise RuntimeError(f"Unexpected Vertex base URL: {BASE_URL}")

user_secrets = UserSecretsClient()
service_account_json = user_secrets.get_secret("GOOGLE_APPLICATION_CREDENTIALS")
os.environ["AIF_GOOGLE_APPLICATION_CREDENTIALS_JSON"] = service_account_json
service_account_info = json.loads(service_account_json)
credentials = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
credentials.refresh(Request())

client = OpenAI(
    api_key=credentials.token,
    base_url=BASE_URL,
)

print("vertex base url:", BASE_URL)
print("vertex model id:", VERTEX_MODEL_ID)
print("vertex credentials: loaded and refreshed")

## 9. Call Qwen through Vertex

This is a one-image endpoint preflight before the full CLI run. The real
evaluation still happens through `aiforensics run --baseline qwen_vl` and
`aiforensics run --baseline assisted_qwen`, which write `predictions.jsonl` for
`evaluate` and `report`.

In [ ]:
# AIF_SECTION: vertex_qwen
import base64
import mimetypes

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def first_image_under(root: Path) -> Path:
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            return path
    raise FileNotFoundError(f"No image found under {root}")


def image_data_url(path: Path) -> str:
    media_type = mimetypes.guess_type(path.name)[0] or "image/png"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{media_type};base64,{encoded}"


VERTEX_QWEN_PROMPT = """You are an image-forensics classifier.

Classify the provided image as either "real" or "fake".
Return exactly one JSON object with keys: label, confidence, evidence.
label must be "real" or "fake"; confidence must be a number from 0 to 1.
"""

VERTEX_TEST_IMAGE = first_image_under(DATA_ROOT)
print("vertex test image:", VERTEX_TEST_IMAGE)

response = client.chat.completions.create(
    model="qwen2_5-vl-7b-instruct-1788570383931",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": VERTEX_QWEN_PROMPT},
                {"type": "image_url", "image_url": {"url": image_data_url(VERTEX_TEST_IMAGE)}},
            ],
        }
    ],
    temperature=0,
    max_tokens=256,
)

print(response.choices[0].message.content)

## 10. Run Qwen-VL across all generators (val split)

These cells execute the pipeline for `qwen_vl` only. Neither CLIP, NPR, nor Assisted Qwen
are executed.

1. `prepare` reads the GenImage layout under `DATA_ROOT`, filters solely the `val` split,
   and writes `genimage_all_val.csv`.
2. `run --baseline qwen_vl` classifies each image in parallel through the Vertex AI Dedicated
   Endpoint.
3. `evaluate` computes classification metrics (accuracy, AUROC, log-loss, etc.) by generator.
4. `report` renders `qwen_vertex_all_val_report.md`.

In [ ]:
# AIF_SECTION: full_run
os.environ["AIF_PREPARE_ARGS"] = "--build-manifests" if BUILD_MANIFESTS else ""
print("prepare args:", os.environ["AIF_PREPARE_ARGS"] or "(validate only)")

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics prepare ${AIF_PREPARE_ARGS} --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics run --baseline qwen_vl --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics evaluate --config "$AIF_CONFIG"

In [ ]:
%%bash
# AIF_SECTION: full_run
set -euo pipefail

cd "$AIF_REPO_ROOT"
aiforensics report --config "$AIF_CONFIG"

## 9. Artifacts

Where to look after a run. The notebook only points at these files; parsing and
rendering stay in the package (`aiforensics evaluate` and `aiforensics report`).

```text
<OUTPUT_ROOT>/manifest_validation.json
<OUTPUT_ROOT>/<run_id>/status.json
<OUTPUT_ROOT>/<run_id>/predictions.jsonl
<OUTPUT_ROOT>/<run_id>/metrics.json
<OUTPUT_ROOT>/<run_id>/metrics_by_source.csv
<OUTPUT_ROOT>/<configured report filename>
```

In [ ]:
report_path = OUTPUT_ROOT / cfg["report"]["filename"]

print("manifest validation:", OUTPUT_ROOT / "manifest_validation.json")
print("report             :", report_path, "exists:", report_path.is_file())

print("\nrun directories under", OUTPUT_ROOT)
if OUTPUT_ROOT.is_dir():
    for entry in sorted(p for p in OUTPUT_ROOT.iterdir() if p.is_dir()):
        print("  -", entry.name)